# Solar Panel Dust Classification — Model Training

Trains two classifiers on the 1280-d EfficientNetB0 features produced by
`solar_preprocessing_pipeline.ipynb` and saved to
`/kaggle/working/outputs/solar_features.pkl`.

**Model A — MLP head (Keras):** the expected best performer. Frozen-backbone
embeddings benefit from a small non-linear head that can combine the 1280
features. We use Dense → Dropout → Dense → softmax with early stopping. No
fine-tuning of EfficientNet itself is needed (or possible here — only the
embeddings are stored).

**Model B — Logistic Regression (sklearn):** the closest classical baseline.
On standardized embeddings it is typically within 1–2 pp of the MLP and is
far cheaper to train. Useful as a sanity check and as a deployable fallback.

In [ ]:
# Cell 1 — Imports
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow {tf.__version__}")

In [ ]:
# Cell 2 — Load features pkl
PKL_PATH    = Path("/kaggle/working/outputs/solar_features.pkl")
OUTPUT_DIR  = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(PKL_PATH, "rb") as f:
    data = pickle.load(f)

X_train, X_val, X_test = data["X_train"], data["X_val"], data["X_test"]
y_train, y_val, y_test = data["y_train"], data["y_val"], data["y_test"]
classes                = data["classes"]
scaler                 = data["scaler"]

n_classes = len(classes)
n_features = X_train.shape[1]

print(f"Classes        : {classes}")
print(f"Feature dim    : {n_features}")
print(f"X_train shape  : {X_train.shape}   y_train shape: {y_train.shape}")
print(f"X_val   shape  : {X_val.shape}     y_val   shape: {y_val.shape}")
print(f"X_test  shape  : {X_test.shape}    y_test  shape: {y_test.shape}")
print(f"Train class counts: {np.bincount(y_train).tolist()}")
print(f"Val   class counts: {np.bincount(y_val).tolist()}")
print(f"Test  class counts: {np.bincount(y_test).tolist()}")

## Model A — MLP head (primary)

A two-layer dense head with dropout and L2 weight decay. Inputs are already
standardized, so no BatchNorm on the input is needed. Training uses early
stopping on val loss and LR reduction on plateau.

In [ ]:
# Cell 3 — Build the MLP head
def build_mlp(input_dim: int, n_classes: int) -> tf.keras.Model:
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.4),
        layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


mlp = build_mlp(n_features, n_classes)
mlp.summary()

In [ ]:
# Cell 4 — Train the MLP
callbacks = [
    EarlyStopping(monitor="val_loss", patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
]

history = mlp.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
# Cell 5 — Training curves
hist = history.history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist["loss"],     label="train")
axes[0].plot(hist["val_loss"], label="val")
axes[0].set_title("MLP loss");     axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist["accuracy"],     label="train")
axes[1].plot(hist["val_accuracy"], label="val")
axes[1].set_title("MLP accuracy"); axes[1].set_xlabel("epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Cell 6 — Evaluate the MLP on test set
mlp_proba = mlp.predict(X_test, verbose=0)
mlp_pred  = mlp_proba.argmax(axis=1)

mlp_acc = accuracy_score(y_test, mlp_pred)
print(f"MLP test accuracy: {mlp_acc:.4f}\n")
print(classification_report(y_test, mlp_pred, target_names=classes, digits=4))

cm = confusion_matrix(y_test, mlp_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=classes)
disp.plot(cmap="Blues", values_format="d")
plt.title("MLP — confusion matrix (test)")
plt.show()

## Model B — Logistic Regression (baseline)

Multinomial logistic regression with L2 regularization. We pick `C` via a tiny
validation sweep rather than full cross-validation so it stays fast.

In [ ]:
# Cell 7 — Tune C on validation set
C_grid = [0.01, 0.1, 1.0, 3.0, 10.0]
best_C, best_val_acc = None, -1.0

for C in C_grid:
    clf = LogisticRegression(
        C=C, max_iter=2000, multi_class="multinomial",
        solver="lbfgs", n_jobs=-1, random_state=42,
    )
    clf.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, clf.predict(X_val))
    print(f"  C={C:<6}  val_acc={val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc, best_C = val_acc, C

print(f"\nBest C = {best_C}  (val_acc={best_val_acc:.4f})")

In [ ]:
# Cell 8 — Refit LR on train+val with the best C, evaluate on test
X_trval = np.vstack([X_train, X_val])
y_trval = np.concatenate([y_train, y_val])

lr = LogisticRegression(
    C=best_C, max_iter=2000, multi_class="multinomial",
    solver="lbfgs", n_jobs=-1, random_state=42,
)
lr.fit(X_trval, y_trval)

lr_pred = lr.predict(X_test)
lr_acc  = accuracy_score(y_test, lr_pred)
print(f"LR test accuracy: {lr_acc:.4f}\n")
print(classification_report(y_test, lr_pred, target_names=classes, digits=4))

cm = confusion_matrix(y_test, lr_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=classes)
disp.plot(cmap="Greens", values_format="d")
plt.title("Logistic Regression — confusion matrix (test)")
plt.show()

## Comparison and persistence

In [ ]:
# Cell 9 — Side-by-side comparison
summary = pd.DataFrame({
    "model":          ["MLP head (Keras)", f"Logistic Regression (C={best_C})"],
    "test_accuracy":  [mlp_acc, lr_acc],
    "n_params":       [int(mlp.count_params()),
                       int(lr.coef_.size + lr.intercept_.size)],
})
print(summary.to_string(index=False))

In [ ]:
# Cell 10 — Save both models
MLP_PATH = OUTPUT_DIR / "mlp_head.keras"
LR_PATH  = OUTPUT_DIR / "logreg.pkl"

mlp.save(MLP_PATH)
with open(LR_PATH, "wb") as f:
    pickle.dump({"model": lr, "best_C": best_C, "classes": classes}, f,
                protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved MLP to {MLP_PATH}  ({MLP_PATH.stat().st_size/1e6:.2f} MB)")
print(f"Saved LR  to {LR_PATH}   ({LR_PATH.stat().st_size/1e6:.2f} MB)")

In [ ]:
# Cell 11 — Reload sanity check: predict a single test sample with each saved model
reloaded_mlp = tf.keras.models.load_model(MLP_PATH)
with open(LR_PATH, "rb") as f:
    reloaded_lr = pickle.load(f)["model"]

sample = X_test[:5]
true   = y_test[:5]

mlp_check = reloaded_mlp.predict(sample, verbose=0).argmax(axis=1)
lr_check  = reloaded_lr.predict(sample)

print("  idx | true | mlp | lr")
print("  ----+------+-----+----")
for i, (t, m, l) in enumerate(zip(true, mlp_check, lr_check)):
    print(f"   {i}  |  {classes[t]:<10s} | {classes[m]:<10s} | {classes[l]}")

print("\nReload sanity check passed.")